In [1]:
import sympy as sym
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from sympy import Q, Matrix

## Introduction
The goal of this program is to symbolicly determine the QFI of a general 2-mode exactly 2-photon state for a pair of loss channels. In order to do this we need to first define our relevant symbols
### Defining the starting state

In [2]:
α,β = sym.symbols('alpha beta',complex=True)
η1,η2 = sym.symbols('eta1 eta2',real=True)

In [3]:
αsq = α*sym.conjugate(α)
βsq = β*sym.conjugate(β)

In [4]:
vec = Matrix([α,β*sym.sqrt(1-αsq),sym.sqrt(1+αsq*βsq-αsq-βsq)])

Note: because sympy is weird about conjugating square roots we need to use a little trickery to avoid that

In [5]:
vecH = Matrix([sym.conjugate(α),sym.conjugate(β)*sym.sqrt(1-αsq),sym.sqrt(1+αsq*βsq-αsq-βsq)]).T

In [6]:
M = sym.simplify(vec@vecH)

In [7]:
M

Matrix([
[                                                                                                alpha*conjugate(alpha),                                                                                                 alpha*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(beta),                                  alpha*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[                                                               beta*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(alpha),                                                                                                      beta*(-alpha*conjugate(alpha) + 1)*conjugate(beta), beta*sqrt(-alpha*conjugate(alpha) + 1)*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)*conjugate(alpha), sqrt(-alpha*conjugate(alpha) + 1)*sq

We now verify that this is indeed a trace-1 operator, and since this is clearly hermitian, it is therefore a valid density operator

In [8]:
sym.simplify(sym.trace(M))

1

In [9]:
assumptions = Q.nonnegative(1-αsq) & Q.nonnegative(1-βsq) & Q.nonnegative(1-η1) & Q.nonnegative(1-η2)

## A comment on the properties of this state
Because this state is in a fixed total photon number state, we can see that clearly $[\hat \rho,\hat N] =0$, or in other words the total excitation operator commutes with the state. For states of this class, under evolution driven by $\mathcal{L}[\hat a_i]$, we find
$[\hat N,\mathcal{L}[\hat a_i]\hat \rho ] = 2[\hat N,\hat a_i \hat \rho \hat a_i^\dagger] -[\hat N,\hat n_i\hat \rho] - [\hat N,\hat \rho\hat n_i]$

$=2[\hat N,\hat a_i] \hat \rho\hat a_i^\dagger + 2\hat a_i[\hat N,\hat \rho \hat a_i^\dagger]$

$=-2\hat a_i \hat \rho\hat a_i^\dagger + 2\hat a_i\hat \rho[\hat N, \hat a_i^\dagger]$

$=-2\hat a_i \hat \rho\hat a_i^\dagger + 2\hat a_i\hat \rho\hat a_i^\dagger$

$=0$

This only used the fact that $[\hat N,\hat \rho]=0$, so therefore $[\hat N,\hat \rho] =0\to[\hat N,\mathcal{L}[\hat a_i]\hat\rho] =0$. This means that under this evolution, for a starting state with fixed photon number, $\hat \rho$ and $\hat N$ must always be simultaneously diagonalizable. In turn this means that we can break our state space into a direct sum of spaces with a fixed number of photons. This trick lets us analyze a trio of matricies of size 1x1,2x2, and 3x3 instead of a single 6x6 matrix. This will be much simpler (and for these 3 sizes sympy should be guaranteed to find an eigenbasis, whereas for larger symbolic matricies it may not find one)

## Evolving the state
In order to evolve the state we look at how each of the three relevant submatricies are constructed.

For the 2 photon subspace, we multiply each coefficient of our original matrix on both left and right by a 3x3 matrix that represents the probability that we will have no transitions

In [10]:
T2to2 = Matrix.diag(η1,η2,sym.sqrt(η1*η2))

In [11]:
T2to2

Matrix([
[eta1,    0,               0],
[   0, eta2,               0],
[   0,    0, sqrt(eta1*eta2)]])

In [12]:
ρ2 = T2to2@M@T2to2

In [13]:
ρ2

Matrix([
[                                                                                                             alpha*eta1**2*conjugate(alpha),                                                                                                            alpha*eta1*eta2*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(beta),                                  alpha*eta1*sqrt(eta1*eta2)*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[                                                                          beta*eta1*eta2*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(alpha),                                                                                                                   beta*eta2**2*(-alpha*conjugate(alpha) + 1)*conjugate(beta), beta*eta2*sqrt(eta1*eta2)*sqrt(-alpha*conjugate(alpha) + 1)*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[eta1*sqrt(eta1*eta2)*sqrt(alph

For the one photon case we now need transition probabilities and the probabilities to remain in that state. First we need the 2x3 matrix corresponding to a each annihilation operator acting on the 2 photon subspace. After this we then multiply by a matrix correzponding to the probability they don't have further transitions.

In [14]:
a12 = Matrix([[sym.sqrt(2),0,0],[0,0,1]])

In [15]:
a22 = Matrix([[0,0,1],[0,sym.sqrt(2),0]])

In [16]:
T1to1 = Matrix.diag(sym.sqrt(η1),sym.sqrt(η2))

In [17]:
ρ2to1 = (1-η1)*a12@M@a12.T + (1-η2)*a22@M@a22.T

In [18]:
ρ1 = sym.simplify(T1to1@ρ2to1@T1to1)

In [19]:
ρ1

Matrix([
[                                                            -eta1*(2*alpha*(eta1 - 1)*conjugate(alpha) + (eta2 - 1)*(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)), sqrt(2)*sqrt(eta1)*sqrt(eta2)*(-alpha*(eta1 - 1) - (eta2 - 1)*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(beta))*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[sqrt(2)*sqrt(eta1)*sqrt(eta2)*(-beta*(eta2 - 1)*sqrt(-alpha*conjugate(alpha) + 1) - (eta1 - 1)*conjugate(alpha))*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1),                                   eta2*(2*beta*(eta2 - 1)*(alpha*conjugate(alpha) - 1)*conjugate(beta) - (eta1 - 1)*(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1))]])

For the vaccuum case we need to now create a final representation that takes states from our two photon state down to zero. For this we need $\hat a_i^2$ and $\hat a_i a_j$ matricies.

In [20]:
a1sq = Matrix([[(1-η1),0,0]])

In [21]:
a2sq = Matrix([[0,(1-η2),0]])

In [22]:
a1a2 = Matrix([[0,0,sym.sqrt(1-η1)*sym.sqrt(1-η2)]])

In [23]:
a1sq@M@a1sq.T

Matrix([[alpha*(1 - eta1)**2*conjugate(alpha)]])

In [24]:
ρ0 = a1sq@M@a1sq.T+a2sq@M@a2sq.T+a1a2@M@a1a2.T

We confirm this is all reasonable by checking the trace of the total state, which again confirms that we have a reasonable state

In [25]:
sym.simplify(sym.trace(ρ0)+sym.trace(ρ1)+sym.trace(ρ2))

1

## Evaluating the Lindbladian
The lindbladian should be key to determining the value of the SLD. This will require a decent amount of computation on each matrix. to this end we will need number operators for each dimension (though for the zero photon state clearly this is just zero) and a new set of annihilators that take the 1 photon terms down to zero photons. Recall
$\mathcal{L}[\hat a_i] \hat \rho = 2\hat a_i\hat\rho\hat a_i^\dagger -\hat\rho\hat n_i -\hat n_i\hat \rho$

In [26]:
n12 = Matrix.diag(2,0,1)
n22 = Matrix.diag(0,2,1)
n11 = Matrix.diag(1,0)
n21 = Matrix.diag(0,1)
a11 = Matrix([[1,0]])
a21 = Matrix([[0,1]])

In [27]:
L12 = -n12@ρ2 -ρ2@n12

In [28]:
L12

Matrix([
[                                                                                                             -4*alpha*eta1**2*conjugate(alpha),                                                                                                          -2*alpha*eta1*eta2*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(beta),                                -3*alpha*eta1*sqrt(eta1*eta2)*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[                                                                          -2*beta*eta1*eta2*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(alpha),                                                                                                                                                                             0, -beta*eta2*sqrt(eta1*eta2)*sqrt(-alpha*conjugate(alpha) + 1)*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[-3*eta1*sqrt(eta1*et

In [29]:
L11 = 2*a12@ρ2@a12.T -n11@ρ1-ρ1@n11

In [30]:
L11

Matrix([
[                                                                                                                                                                                 4*alpha*eta1**2*conjugate(alpha) + 2*eta1*(2*alpha*(eta1 - 1)*conjugate(alpha) + (eta2 - 1)*(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)), 2*sqrt(2)*alpha*eta1*sqrt(eta1*eta2)*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1) - sqrt(2)*sqrt(eta1)*sqrt(eta2)*(-alpha*(eta1 - 1) - (eta2 - 1)*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(beta))*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)],
[-sqrt(2)*sqrt(eta1)*sqrt(eta2)*(-beta*(eta2 - 1)*sqrt(-alpha*conjugate(alpha) + 1) - (eta1 - 1)*conjugate(alpha))*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1) + 2*sqrt(2)*eta1*sqrt(eta1*eta2)*sqrt(alpha*be

In [31]:
L10 = 2*a11@ρ1@a11.T

In [32]:
L10

Matrix([[-2*eta1*(2*alpha*(eta1 - 1)*conjugate(alpha) + (eta2 - 1)*(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1))]])

In [33]:
L22 = -n22@ρ2 -ρ2@n22

In [34]:
L21 = 2*a22@ρ2@a22.T -n21@ρ1-ρ1@n21

In [35]:
L20 = 2*a21@ρ1@a21.T

In [36]:
ρ2.eigenvects()

[(0,
  2,
  [Matrix([
   [-eta2*sqrt(-alpha*conjugate(alpha) + 1)*conjugate(beta)/(eta1*conjugate(alpha))],
   [                                                                              1],
   [                                                                              0]]),
   Matrix([
   [-sqrt(eta1*eta2)*sqrt(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1)/(eta1*conjugate(alpha))],
   [                                                                                                                                             0],
   [                                                                                                                                             1]])]),
 (alpha*eta1**2*conjugate(alpha) - beta*eta2**2*(alpha*conjugate(alpha) - 1)*conjugate(beta) + eta1*eta2*(alpha*beta*conjugate(alpha)*conjugate(beta) - alpha*conjugate(alpha) - beta*conjugate(beta) + 1),
  1,
  [Matrix([
   [                              